In [18]:
# ======= Variables =========

# Fill in the variables below and run cell first
# Note: Use forward slashes / instead of backslashes \ in file paths

DATASET_DIR = "../dataset"
OUTPUT_DIR = f"{DATASET_DIR}/original_sheets"

MODEL_TABLE_FILENAME = "model_table.csv"
DIAGNOSIS_DATASET_FILENAME = "diagnosis_dataset.csv"
OUTPUT_MODEL_TABLE_FILENAME = "model_table_feature_engineered.csv"

In [19]:
# Read the model_table and diagnosis_dataset DataFrames

import pandas as pd

model_table_df = pd.read_csv(f"{DATASET_DIR}/{MODEL_TABLE_FILENAME}")
diagnosis_dataset_df = pd.read_csv(f"{DATASET_DIR}/{DIAGNOSIS_DATASET_FILENAME}")

print(f"Model table: rows={model_table_df.shape[0]}, columns={model_table_df.shape[1]}")
print(f"Diagnosis dataset: rows={diagnosis_dataset_df.shape[0]}, columns={diagnosis_dataset_df.shape[1]}")

Model table: rows=4020, columns=9
Diagnosis dataset: rows=4020, columns=4


In [4]:
# Explore unique values for model table
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
from ml_antiviral_diagnosis import summarize_unique_values

model_table_eda_df = summarize_unique_values(model_table_df)
model_table_eda_df

,column,dtype,unique_count,unique_values
0,PATIENT_ID,int64,4020,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14..."
1,TARGET,int64,2,"[0, 1]"
2,DISEASEX_DT,str,77,"[2022-06-11, 2022-06-22, 2022-06-20, 2022-06-3..."
3,PATIENT_AGE,int64,86,"[34, 2, 49, 0, 71, 1, 45, 31, 74, 32, 23, 82, ..."
4,PATIENT_GENDER,str,2,"[M, F]"
5,NUM_CONDITIONS,int64,7,"[0, 2, 1, 3, 4, 6, 5]"
6,PHYSICIAN_TYPE,str,78,"[FAMILY MEDICINE, EMERGENCY MEDICINE, PEDIATRI..."
7,PHYSICIAN_STATE,str,53,"[TX, PA, MS, nan, CA, WV, ME, FL, AL, AR, MD, ..."
8,LOCATION_TYPE,str,28,"[EMERGENCY ROOM - HOSPITAL, OFFICE, INDEPENDEN..."


In [11]:
# Peruse the unique values of a specific column
column_name = "LOCATION_TYPE" #"PHYSICIAN_TYPE"
values = model_table_eda_df.loc[
    model_table_eda_df["column"] == column_name, "unique_values"
].iloc[0]
values

['EMERGENCY ROOM - HOSPITAL',
 'OFFICE',
 'INDEPENDENT LABORATORY',
 'INPATIENT HOSPITAL',
 'HOSPITAL OUTPATIENT',
 'UNASSIGNED',
 'CLINIC - FREESTANDING',
 'URGENT CARE FACILITY',
 'AMBULANCE - LAND',
 "TELEHEALTH PROVIDED OTHER THAN IN PATIENT'S HOME",
 "TELEHEALTH PROVIDED IN PATIENT'S HOME",
 'WALK-IN RETAIL HEALTH CLINIC',
 'OTHER PLACE OF SERVICE',
 'HOSPITAL INPATIENT (INCLUDING MEDICARE PART A)',
 'FEDERALLY QUALIFIED HEALTH CENTER',
 'CLINIC - RURAL HEALTH',
 'CLINIC  - FEDERALLY QUALIFIED HEALTH CENTER (FQHC)',
 'ON CAMPUS-OUTPATIENT HOSPITAL',
 'HOSPITAL - LABORATORY SERVICES PROVIDED TO NON-PATIENTS',
 'OFF CAMPUS-OUTPATIENT HOSPITAL',
 'RURAL HEALTH CLINIC',
 'HOSPITAL INPATIENT (MEDICARE PART B ONLY)',
 'INPATIENT PSYCHIATRIC FACILITY',
 'PHARMACY',
 'CRITICAL ACCESS HOSPITAL',
 'COMPREHENSIVE OUTPATIENT REHABILITATION FACILITY',
 'AMBULANCE - AIR OR WATER',
 'MOBILE UNIT']

In [20]:
# Add transaction-based features to the model table
import importlib
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import ml_antiviral_diagnosis.feature_engineering as feature_engineering_module

importlib.reload(feature_engineering_module)

feature_engineered_model_table_df = feature_engineering_module.add_model_table_transaction_features(
    model_table_df=model_table_df,
    diagnosis_dataset_df=diagnosis_dataset_df,
 )

print(
    f"Feature-engineered model table: rows={feature_engineered_model_table_df.shape[0]}, "
    f"columns={feature_engineered_model_table_df.shape[1]}"
 )
feature_engineered_model_table_df[[
    "PATIENT_ID",
    "INSURANCE_TYPE",
    "CONTRAINDICATIONS",
    "HIGH_RISK",
]].head()

Feature-engineered model table: rows=4020, columns=12


,PATIENT_ID,INSURANCE_TYPE,CONTRAINDICATIONS,HIGH_RISK
0,1,COMMERCIAL,Unspecified,0
1,2,COMMERCIAL,Unspecified,0
2,3,COMMERCIAL,Unspecified,0
3,4,COMMERCIAL,Unspecified,0
4,5,COMMERCIAL,Unspecified,0


In [29]:
# Explore unique values for feature_engineered_model_table_df
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
from ml_antiviral_diagnosis import summarize_unique_values

feature_engineered_model_table_eda_df = summarize_unique_values(feature_engineered_model_table_df)
feature_engineered_model_table_eda_df

,column,dtype,unique_count,unique_values
0,PATIENT_ID,int64,4020,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14..."
1,TARGET,int64,2,"[0, 1]"
2,DISEASEX_DT,str,77,"[2022-06-11, 2022-06-22, 2022-06-20, 2022-06-3..."
3,PATIENT_AGE,int64,86,"[34, 2, 49, 0, 71, 1, 45, 31, 74, 32, 23, 82, ..."
4,PATIENT_GENDER,str,2,"[M, F]"
5,NUM_CONDITIONS,int64,7,"[0, 2, 1, 3, 4, 6, 5]"
6,PHYSICIAN_TYPE,str,77,"[FAMILY MEDICINE, EMERGENCY MEDICINE, PEDIATRI..."
7,PHYSICIAN_STATE,str,53,"[TX, PA, MS, UNSPECIFIED, CA, WV, ME, FL, AL, ..."
8,LOCATION_TYPE,str,28,"[EMERGENCY ROOM - HOSPITAL, OFFICE, INDEPENDEN..."
9,INSURANCE_TYPE,str,4,"[COMMERCIAL, MEDICARE, UNSPECIFIED, MEDICAID]"


In [22]:
# Clean missing categorical values, verify null counts, and save the feature-engineered model table
import importlib
import ml_antiviral_diagnosis.feature_engineering as feature_engineering_module

importlib.reload(feature_engineering_module)

feature_engineered_model_table_df = feature_engineering_module.clean_model_table_categorical_nulls(
    feature_engineered_model_table_df,
    fill_value="UNSPECIFIED",
 )

null_counts = feature_engineered_model_table_df.isna().sum()
null_counts

output_path = f"{DATASET_DIR}/{OUTPUT_MODEL_TABLE_FILENAME}"
feature_engineered_model_table_df.to_csv(output_path, index=False)

print(f"Saved feature-engineered model table to {output_path}")

Saved feature-engineered model table to ../dataset/model_table_feature_engineered.csv


In [28]:
# Inspect the HIGH_RISK distribution
feature_engineered_model_table_df["HIGH_RISK"].value_counts(dropna=False).sort_index()

HIGH_RISK
0    1513
1    2507
Name: count, dtype: int64